# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.

## Setup

In [47]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [48]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput

## Understanding the Agent Framework Architecture

The Microsoft Agent Framework follows a layered architecture:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – An `AIProjectClient` connects to an Azure OpenAI deployment. It handles authentication, request formatting, and response parsing.
2. **Agent** – Created from the client via `project_client.agents.create_version()`, the agent combines model access with instructions (system prompt) and tools.
3. **Tools** – Python functions decorated with `FunctionTool` that the agent can invoke to perform actions or retrieve data.
4. **Session** – An `AgentSession` object (created via `agent.create_session()`) that stores conversation history, enabling multi-turn dialogue where the agent remembers prior context.

Let's build each layer step by step.

In [49]:
# Create the client – this is the connection to the AI model
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

## Adding Tools with the @tool Decorator

Tools let agents take actions beyond generating text. The `@tool` decorator converts a regular Python function into something the agent can call.

Key points:
- Use `Annotated[type, "description"]` so the model understands each parameter.
- The docstring becomes the tool description the model sees.
- `approval_mode="never_require"` means the tool runs automatically without user confirmation.

In [50]:
def check_destination_availability(
    destination: str
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creating an Agent with Tools

Now we combine the client, instructions, and tools into an agent. The `instructions` act as the system prompt — they define the agent's persona and behaviour.

In [51]:
destination_tool = FunctionTool(
    name="check_destination_availability",
    description="Check if a vacation destination is currently available for booking.",
    parameters= {
        "type": "object",
        "properties": {
          "destination": { "type": "string" }
        },
        "required": ["destination"]
      }
)

tools: list[Tool] = [destination_tool]

agent = project_client.agents.create_version(
    agent_name="TravelAvailabilityAgent",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=(
            "You are a travel booking agent. Help users check destination availability "
            "and make recommendations. Always check availability before recommending a destination."
        ),
        tools=tools
    )
)

print(f"Created agent: name={agent.name}, version={agent.version}")

Created agent: name=TravelAvailabilityAgent, version=2


## Multi-Turn Conversations with Sessions

An `AgentSession` (created via `agent.create_session()`) keeps track of all messages in a conversation. By passing the same session to each `agent.run()` call, the agent has access to the full conversation history and can refer back to earlier messages.

We pass `tools=[check_destination_availability]` so the agent can call our availability checker during each turn.

In [52]:
openai = project_client.get_openai_client()

# Start a multi-turn conversation
conversation = openai.conversations.create()
print(f"Conversation created: {conversation.id}")

# Turn 1: Ask about available destinations
response = openai.responses.create(
    input="Can I vacation in Barcelona?",
    conversation=conversation.id,
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)

tool_outputs = []

# Handle any requested function calls
for item in response.output:
    if item.type == "function_call":
        if item.name == "check_destination_availability":
            args = json.loads(item.arguments)
            print(f"Args: {args['destination']}")
            result = check_destination_availability(args['destination'])

            tool_outputs.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"result": result}),
                )
            )

# If the agent requested a tool, send the tool output back
if tool_outputs:
    final_response = openai.responses.create(
        input=tool_outputs,
        previous_response_id=response.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print("Final agent response:")
    print(final_response.output_text)
else:
    print("Agent response:")
    print(response.output_text)

response = openai.responses.create(
            conversation=conversation.id,
            input=tool_outputs,
            extra_body={
                "agent_reference": {
                    "name": agent.name,
                    "type": "agent_reference",
                }
            },
        )

Conversation created: conv_3dff5457d9cba30400Yeifxrqd4P22APPh8kX0TkNlsBANr0Hi
Args: Barcelona
Final agent response:
Yes, Barcelona is available for booking! Would you like any recommendations on what to do there?


In [53]:
# Turn 2: Follow-up question — the agent remembers the conversation

follow_up = openai.responses.create(
    input="Can I eat pizz today?",
    conversation=conversation.id,
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)
    
tool_outputs = []

# Handle any requested function calls
for item in follow_up.output:
    if item.type == "function_call":
        if item.name == "check_destination_availability":
            args = json.loads(item.arguments)
            print(f"Args: {args['destination']}")
            result = check_destination_availability(args['destination'])

            tool_outputs.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"result": result}),
                )
            )

# If the agent requested a tool, send the tool output back
if tool_outputs:
    final_response = openai.responses.create(
        input=tool_outputs,
        previous_response_id=follow_up.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print("Final agent response:")
    print(final_response.output_text)
else:
    print("Agent response:")
    print(follow_up.output_text)

follow_up = openai.responses.create(
            conversation=conversation.id,
            input=tool_outputs,
            extra_body={
                "agent_reference": {
                    "name": agent.name,
                    "type": "agent_reference",
                }
            },
        )

Agent response:
It seems like there might be a typo. If you're asking about eating pizza today, you can certainly find places to enjoy some pizza! If you meant something else, please let me know.


## Summary

In this lesson you explored the four pillars of the Microsoft Agent Framework:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` connects to Azure OpenAI with credential-based auth |
| **Agent** | `provider.create_agent()` bundles a model connection with instructions and a name |
| **Tools** | The `@tool` decorator exposes Python functions for the agent to call |
| **Session** | `agent.create_session()` maintains conversation history across multiple turns |

These building blocks compose together to create agents that can hold natural conversations, call external functions, and maintain context — the foundation for more advanced agentic patterns in later lessons.